In [5]:
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from typing import Literal, TypedDict
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt


class ReviewState(TypedDict):
    content: str
    approved: bool
    feedback: str


def generate_content(state: ReviewState) -> dict:
    """生成内容"""
    print("📝 生成内容...")
    return {"content": "这是生成的内容..."}


def review_node(state: ReviewState) -> dict:
    """人工审核节点"""
    content = state["content"]

    print(f"\n{'=' * 50}")
    print("⏸️  等待人工审核...")
    print(f"内容: {content}")
    print(f"{'=' * 50}\n")

    # 暂停并等待人工输入
    approval = interrupt({
        "question": "是否批准此内容？",
        "options": ["approve", "reject", "request_changes"]
    })

    # 根据审核结果返回
    if approval == "approve":
        return {"approved": True, "feedback": "已批准"}
    elif approval == "reject":
        return {"approved": False, "feedback": "已拒绝"}
    else:
        return {"approved": False, "feedback": "需要修改"}


# 这个函数的返回值似乎不合法
def review_with_options(state: ReviewState) -> dict:
    """带多个选项的审核"""
    content = state["content"]

    # 提供结构化的选项
    decision = interrupt({
        "type": "review",
        "content": content,
        "options": [
            {"id": "approve", "label": "批准", "description": "内容无问题，批准发布"},
            {"id": "minor_edit", "label": "小修改", "description": "需要小幅修改"},
            {"id": "major_edit", "label": "大修改", "description": "需要重写"},
            {"id": "reject", "label": "拒绝", "description": "内容不合适，拒绝"}
        ]
    })

    return {"decision": decision}


def publish_node(state: ReviewState) -> dict:
    """发布内容"""
    print("🚀 发布内容...")
    return {}


def reject_node(state: ReviewState) -> dict:
    """拒绝内容"""
    print("❌ 内容已拒绝")
    return {}


def should_publish(state: ReviewState) -> Literal["publish", "reject"]:
    """根据审核结果路由"""
    return "publish" if state.get("approved", False) else "reject"


# 构建图（必须使用 checkpointer）
graph = StateGraph(ReviewState)
graph.add_node("generate", generate_content)
graph.add_node("review", review_node)
graph.add_node("publish", publish_node)
graph.add_node("reject", reject_node)

graph.add_edge(START, "generate")
graph.add_edge("generate", "review")
graph.add_conditional_edges(
    "review",
    should_publish,
    {
        "publish": "publish",
        "reject": "reject"
    }
)

graph.add_edge("publish", END)
graph.add_edge("reject", END)

In [10]:
from LangGraph.envInit import display_graph

# 必须使用 checkpointer
checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)
display_graph(app)

# 运行
config = {"configurable": {"thread_id": "review-1"}}

print("\n=== 第一次调用（会中断）===")
result = app.invoke({"content": "", "approved": False}, config)
print(f"状态: {result}")

# 检查是否中断
state = app.get_state(config)
print(f"\n中断状态: {state.next}")  # 应该显示等待继续的节点

# 提供审核决策并继续
print("\n=== 提供审核决策并继续 ===")
app.update_state(config, {"approved": True})  # 批准
result = app.invoke(None, config)  # 继续执行
print(f"最终结果: {result}")


=== 第一次调用（会中断）===
📝 生成内容...

⏸️  等待人工审核...
内容: 这是生成的内容...

状态: {'content': '这是生成的内容...', 'approved': False, '__interrupt__': [Interrupt(value={'question': '是否批准此内容？', 'options': ['approve', 'reject', 'request_changes']}, id='c49dbedb49acec9cdff562a54bffac1a')]}

中断状态: ('review',)

=== 提供审核决策并继续 ===

⏸️  等待人工审核...
内容: 这是生成的内容...

最终结果: {'content': '这是生成的内容...', 'approved': True, '__interrupt__': [Interrupt(value={'question': '是否批准此内容？', 'options': ['approve', 'reject', 'request_changes']}, id='bae7bb766a71bed6b34a48dede229ac4')]}
